In [1]:
import requests
from dotenv import load_dotenv
import os
import json
import time
from typing import Dict

load_dotenv()

api_key = os.getenv("RAPID_API_KEY")

In [2]:
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    listings = json.load(f)

ids = [listing["id"] for listing in listings if "id" in listing]

print(f"Extracted {len(ids)} listing IDs")

# check if ids are unique
if len(ids) == len(set(ids)):
    print("Ids are all unique")
else:
    print("IDS ARE NOT UNIQUE. PLEASE CORRECT BEFORE GETTING APARTMENT DETAILS")

Extracted 6447 listing IDs
Ids are all unique


In [3]:
def load_existing_details() -> Dict:
    """Load existing apartment details."""
    if os.path.exists("manhattan_details.json"):
        with open("manhattan_details.json", "r", encoding="utf-8") as f:
            details = json.load(f)
            return {str(item["id"]): item for item in details}
    return {}

# Replace the second cell with:
# Load changed listings
with open("changed_listings.json", "r", encoding="utf-8") as f:
    changed_listings = json.load(f)

# Load current listings to know what should be deleted
with open("manhattan_listings.json", "r", encoding="utf-8") as f:
    current_listings = json.load(f)
    current_ids = set(str(listing["id"]) for listing in current_listings)

# Load existing details
existing_details = load_existing_details()

# Identify listings to process
new_ids = [str(listing["id"]) for listing in changed_listings]
print(f"Found {len(new_ids)} listings to update")

# Identify listings to remove
removed_ids = set(existing_details.keys()) - current_ids
if removed_ids:
    print(f"Found {len(removed_ids)} listings to remove")

Found 2475 listings to update
Found 2666 listings to remove


In [4]:
headers = {
    "x-rapidapi-key": api_key,
    "x-rapidapi-host": "streeteasy-api.p.rapidapi.com"
}

# Fetch details for new/updated listings
for i, listing_id in enumerate(new_ids, 1):
    url = f"https://streeteasy-api.p.rapidapi.com/rentals/{listing_id}"
    try:
        r = requests.get(url, headers=headers, timeout=30)
        r.raise_for_status()
        existing_details[listing_id] = r.json()
        print(f"Fetched {i}/{len(new_ids)}: {listing_id}")
    except requests.RequestException as e:
        print(f"Error fetching {listing_id}: {e}")
        continue
    time.sleep(0.2)

# Remove listings that no longer exist
for removed_id in removed_ids:
    existing_details.pop(removed_id, None)

# Save updated details
with open("manhattan_details.json", "w", encoding="utf-8") as f:
    json.dump(list(existing_details.values()), f, indent=2)

print(f"Updated details: {len(new_ids)} new/changed, {len(removed_ids)} removed")

Fetched 1/2475: 4888445
Fetched 2/2475: 4888443
Fetched 3/2475: 4888441
Fetched 4/2475: 4888435
Fetched 5/2475: 4888434
Fetched 6/2475: 4888431
Fetched 7/2475: 4888426
Fetched 8/2475: 4888420
Fetched 9/2475: 4888418
Fetched 10/2475: 4888416
Fetched 11/2475: 4888415
Fetched 12/2475: 4888414
Fetched 13/2475: 4888411
Fetched 14/2475: 4888410
Fetched 15/2475: 4888403
Fetched 16/2475: 4888392
Fetched 17/2475: 4888390
Fetched 18/2475: 4888365
Fetched 19/2475: 4888364
Fetched 20/2475: 4888363
Fetched 21/2475: 4888362
Fetched 22/2475: 4888361
Fetched 23/2475: 4888358
Fetched 24/2475: 4888355
Fetched 25/2475: 4888352
Fetched 26/2475: 4888351
Fetched 27/2475: 4888345
Fetched 28/2475: 4888344
Fetched 29/2475: 4888343
Fetched 30/2475: 4888335
Fetched 31/2475: 4888330
Fetched 32/2475: 4888328
Fetched 33/2475: 4888325
Fetched 34/2475: 4888324
Fetched 35/2475: 4888320
Fetched 36/2475: 4888319
Fetched 37/2475: 4888306
Fetched 38/2475: 4888305
Fetched 39/2475: 4888298
Fetched 40/2475: 4888295
Fetched 4

In [5]:
# converts json to csv, no longer needed


# with open("manhattan_details.json", "r") as f:
#     data = json.load(f)

# df = pd.json_normalize(
#     data,
#     sep="_",  # replaces nested keys with underscore, e.g. building_id
# )

# # Convert list-type columns to comma-separated strings
# for col in df.columns:
#     df[col] = df[col].apply(
#         lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x
#     )

# # Save to CSV
# df.to_csv("manhattan_details.csv", index=False)
# print("json to csv conversion successful")